# ROGII - blend our file21 (6.689) with forked file44 (6.464)

Two independent pipeline lineages. Mean abs disagreement 1.669 ft on the real test set. BLEND_W=0.50 as a starting point (no local ground truth to tune against).

Run: Input = competition dataset + rogii-blend-p21-p44 (private). CPU. Internet off. Run All.


In [ ]:
"""Blend our own best pipeline (file21, 6.689 real LB) with the forked public-base+WARP pipeline
(file44, 6.464 real LB) -- two genuinely different pipeline lineages (ours: PF+beam+GRU-refiner+
reversal; theirs: PF+contact-override+model-package+hedge+WARP). Mean abs disagreement between the two
on the real test set: 1.669 ft (median 1.439, max 6.14) -- substantial, suggesting real decorrelation
potential. BLEND_W is the weight on p44 (the currently-stronger individual pipeline); (1-BLEND_W) on p21.
"""
import glob, os, pandas as pd

BLEND_W = 0.50  # weight on p44; (1-BLEND_W) on p21

_assets = glob.glob('/kaggle/input/**/p21_submission.csv', recursive=True)
assert _assets, 'p21_submission.csv not found -- attach the rogii-blend-p21-p44 dataset'
ASSET_DIR = os.path.dirname(_assets[0])

_comp = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
assert _comp, 'sample_submission.csv not found -- attach the competition dataset'
SAMPLE = _comp[0]

p21 = pd.read_csv(f'{ASSET_DIR}/p21_submission.csv', dtype={'id': 'string'})
p44 = pd.read_csv(f'{ASSET_DIR}/p44_submission.csv', dtype={'id': 'string'})
sample = pd.read_csv(SAMPLE, dtype={'id': 'string'})[['id']]

p21 = sample.merge(p21, on='id', how='left')
p44 = sample.merge(p44, on='id', how='left')
assert p21['tvt'].notna().all() and p44['tvt'].notna().all(), 'missing ids after merge'
assert (p21['id'] == p44['id']).all() and (p21['id'] == sample['id']).all()

blended = sample.copy()
blended['tvt'] = (1.0 - BLEND_W) * p21['tvt'].to_numpy(float) + BLEND_W * p44['tvt'].to_numpy(float)

assert len(blended) == len(sample)
assert blended['tvt'].notna().all()
blended[['id', 'tvt']].to_csv('submission.csv', index=False)
print(f'blend_w(p44)={BLEND_W}  rows={len(blended)}  '
      f'tvt mean={blended["tvt"].mean():.3f} std={blended["tvt"].std():.3f}')
print('saved submission.csv')

